# Notebook 10 — Simplify the classified bike network

Turns the lab-classified bike-infrastructure layer into a compact GeoJSON the dashboard embeds on the
Home map. The source is ~90 MB (full-precision coordinates); this shrinks it to a few MB via
(1) Douglas–Peucker simplification and (2) coordinate rounding, while keeping every segment.

**Input** (large, not committed — placed under `data/` by the user):
- `data/Bike_infra_4class_4326.geojson` — every segment carries the lab classification
  (`class_no` 1–4, `bike_class`, `class_name`), derived from the German `radweg_art` attribute.

**Output** (small, committed to `output/`):
- `bike_network_simplified.geojson` — geometry + `class_no` + `class_name` only.

**Notes:**
- CRS is WGS84 (EPSG:4326); coordinates are [lon, lat].
- In the dashboard the four classes are coloured by SAFETY rank, not by class number:
  Class I = safest, then IV, then II, then III = least safe.
- Streamed with `ijson` so the 90 MB file is never fully loaded into memory.

In [1]:
import ijson
import json
from collections import Counter
from shapely.geometry import LineString

NETWORK_GEOJSON = "data/Bike_infra_4class_4326.geojson"
OUT_GEOJSON     = "output/bike_network_simplified.geojson"

SIMPLIFY_TOL_DEG = 0.00012   # ~13 m at Hamburg's latitude
COORD_DECIMALS   = 5         # ~1.1 m -- plenty for a city-scale overview map

# --- added: coarse grey "context" network + simplified city boundary (for dashboard underlays) ---
BOUNDARY_IN   = "data/HH_Boundary.geojson"
OUT_CONTEXT   = "output/bike_network_context.geojson"     # linemerged + coarse, all 4 classes, grey underlay
OUT_BOUNDARY  = "output/hamburg_boundary_simplified.geojson"
CONTEXT_SIMPLIFY_DEG  = 0.0004   # ~40 m -- coarse; this is a faint reference layer
BOUNDARY_SIMPLIFY_DEG = 0.0008   # ~80 m -- city outline
CONTEXT_DECIMALS      = 4        # ~11 m -- fine for a grey context layer


In [2]:
out_features = []
n_in = skipped = pts_before = pts_after = 0
by_class = Counter()

with open(NETWORK_GEOJSON, "rb") as f:
    for feat in ijson.items(f, "features.item"):
        n_in += 1
        geom = feat.get("geometry") or {}
        if geom.get("type") != "LineString":
            skipped += 1
            continue
        coords = [(float(x), float(y)) for x, y in geom.get("coordinates", [])]
        if len(coords) < 2:
            skipped += 1
            continue
        props = feat.get("properties") or {}
        cls = props.get("class_no")
        if cls not in (1, 2, 3, 4):
            skipped += 1
            continue
        pts_before += len(coords)
        simp = LineString(coords).simplify(SIMPLIFY_TOL_DEG, preserve_topology=False)
        cc = [[round(x, COORD_DECIMALS), round(y, COORD_DECIMALS)] for x, y in simp.coords]
        pts_after += len(cc)
        by_class[int(cls)] += 1
        out_features.append({
            "type": "Feature",
            "geometry": {"type": "LineString", "coordinates": cc},
            "properties": {"class_no": int(cls)},
        })

print(f"features in: {n_in:,}  kept: {len(out_features):,}  skipped: {skipped}")
print(f"coordinate points: {pts_before:,} -> {pts_after:,}  ({100 * pts_after / pts_before:.0f}% kept)")
print("kept by class_no:", dict(sorted(by_class.items())))

features in: 94,285  kept: 94,275  skipped: 10
coordinate points: 432,650 -> 202,920  (47% kept)
kept by class_no: {1: 13871, 2: 6278, 3: 53331, 4: 20795}


In [3]:
fc = {"type": "FeatureCollection", "features": out_features}
with open(OUT_GEOJSON, "w") as f:
    json.dump(fc, f, separators=(",", ":"))   # minified

import os
print(f"Wrote {OUT_GEOJSON} ({os.path.getsize(OUT_GEOJSON) / 1e6:.2f} MB)")

Wrote output/bike_network_simplified.geojson (12.97 MB)


## Context network + city boundary (for the dashboard map underlays)

Two extra small outputs used only for visual context on the dashboard maps:
- `bike_network_context.geojson` — the whole network, **linemerged then coarsely simplified**, drawn as one faint grey layer under the station dots on the seasonal maps. Linemerge first (then simplify) is what actually reduces the point count, because the raw file is ~94k short segments.
- `hamburg_boundary_simplified.geojson` — the Hamburg city outline (from `HH_Boundary.geojson`, EPSG:4326), simplified to a light outline. Two parts: the main city area and the Neuwerk exclave.

In [4]:
# --- coarse grey context network (linemerge per class, then simplify) ---
from shapely.geometry import LineString, MultiLineString
from shapely.ops import linemerge

_byclass = {1: [], 2: [], 3: [], 4: []}
with open(NETWORK_GEOJSON, "rb") as f:
    for feat in ijson.items(f, "features.item"):
        g = feat.get("geometry") or {}
        if g.get("type") != "LineString":
            continue
        cls = (feat.get("properties") or {}).get("class_no")
        if cls not in (1, 2, 3, 4):
            continue
        coords = [(float(x), float(y)) for x, y in g["coordinates"]]
        if len(coords) >= 2:
            _byclass[int(cls)].append(LineString(coords))

def _round_geom(geom, nd):
    if geom.is_empty:
        return []
    lines = [geom] if geom.geom_type == "LineString" else list(geom.geoms)
    return [[[round(x, nd), round(y, nd)] for x, y in ls.coords] for ls in lines]

ctx_features = []
ctx_pts = 0
for cls in (1, 2, 3, 4):
    merged = linemerge(MultiLineString(_byclass[cls]))
    simp = merged.simplify(CONTEXT_SIMPLIFY_DEG, preserve_topology=False)
    lines = _round_geom(simp, CONTEXT_DECIMALS)
    ctx_pts += sum(len(l) for l in lines)
    ctx_features.append({
        "type": "Feature",
        "geometry": {"type": "MultiLineString", "coordinates": lines},
        "properties": {"class_no": cls},
    })

with open(OUT_CONTEXT, "w") as f:
    json.dump({"type": "FeatureCollection", "features": ctx_features}, f, separators=(",", ":"))

import os
print(f"Wrote {OUT_CONTEXT} ({os.path.getsize(OUT_CONTEXT)/1e6:.2f} MB, {ctx_pts:,} points)")

Wrote output/bike_network_context.geojson (2.27 MB, 123,974 points)


In [5]:
# --- simplified city boundary outline ---
with open(BOUNDARY_IN) as f:
    _bnd = json.load(f)

def _rings(geom):
    t = geom["type"]; c = geom["coordinates"]
    if t == "Polygon":
        return c
    if t == "MultiPolygon":
        return [ring for poly in c for ring in poly]
    return []

bnd_lines = []
bnd_pts = 0
for feat in _bnd.get("features", []):
    for ring in _rings(feat["geometry"]):
        ls = LineString([(float(x), float(y)) for x, y in ring])
        s = ls.simplify(BOUNDARY_SIMPLIFY_DEG, preserve_topology=False)
        coords = [[round(x, CONTEXT_DECIMALS), round(y, CONTEXT_DECIMALS)] for x, y in s.coords]
        if len(coords) >= 2:
            bnd_lines.append(coords); bnd_pts += len(coords)

with open(OUT_BOUNDARY, "w") as f:
    json.dump({"type": "FeatureCollection",
               "features": [{"type": "Feature",
                             "geometry": {"type": "MultiLineString", "coordinates": bnd_lines},
                             "properties": {"name": "Hamburg boundary"}}]},
              f, separators=(",", ":"))

print(f"Wrote {OUT_BOUNDARY} ({os.path.getsize(OUT_BOUNDARY)/1e3:.0f} KB, {len(bnd_lines)} rings, {bnd_pts:,} points)")

Wrote output/hamburg_boundary_simplified.geojson (15 KB, 10 rings, 880 points)
